# Open-shell and large-amplitude-motion models

A thin driver over three models, each with exact limits tested in `tests/python/test_m8_open_shell_lam.py`:

* `open_rotor`: an asymmetric rotor with electron spin S. It covers spin-rotation, spin-spin and their distortion terms, half-integer J (Kramers class), and Hund's case (b) labels from ⟨N²⟩. It is validated against CDMS O₂ to 5×10⁻⁷ cm⁻¹ (`gold/cdms-o2-v0`).
* `inversion`: a double-well large-amplitude coordinate (NH₃-type) with parity blocks and inversion-rotation coupling.
* `ram_torsion`: internal rotation by the rho-axis method in a full asymmetric rotor (K mixed).

The parameters are **illustrative** unless stated otherwise.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import riss_drucker as rd
WORK = Path(os.environ.get("RISS_DRUCKER_WORK", "/kaggle/working" if Path("/kaggle/working").is_dir() else "."))
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
except ImportError:
    plt = None

## 1. Doublet asymmetric rotor: spin-rotation fine structure (F₁ / F₂)

In [ ]:
m = rd.compile_open_rotor(two_s=1, n_max=12, order=2)
print([(c["name"], c["passed"]) for c in m.compile_certificates])
th = {"A": 8.0, "B": 0.43, "C": 0.41, "eps_aa": 0.18, "eps_bb": 0.0017, "eps_cc": -0.0003}  # NO2-like magnitudes
sol = rd.solve(m, th, window=(-1e9, 1e9), method="dense", keep_vectors=True)
lab = rd.open_shell.assign(m, sol)
lv = {}
for b, r in zip(sol.blocks, sol.results):
    for e, l in zip(r["eigenvalues"], lab[b["key"]]):
        lv.setdefault((l["N"], l["Ka"], l["F"]), []).append(e)
print("half-integer J blocks:", [b["two_j"] / 2 for b in sol.blocks][:5], "...")
for K in (0, 1, 2):
    s = [(N, np.mean(lv[(N, K, 1)]) - np.mean(lv[(N, K, 2)])) for N in range(max(K, 1), 11) if (N, K, 1) in lv and (N, K, 2) in lv]
    print(f"Ka={K}: F1-F2 splitting (cm-1) for N=1..:", np.round([x for _, x in s], 4))

## 2. Triplet: O₂ X³Σg⁻ against CDMS (runs when the files are present)

In [ ]:
o2 = Path(os.environ.get("RISS_DRUCKER_DATA", "/kaggle/input/riss-drucker-data")) / "o2_cdms"
if (o2 / "v0.var").exists():
    import subprocess, sys
    print(subprocess.run([sys.executable, "scripts/gold_cdms_o2.py", "--data", str(o2), "--out", str(WORK / "o2_gold")],
                         capture_output=True, text=True).stdout)
else:
    print("O2 CDMS files not present; see gold/cdms-o2-v0/record.json for the validated result")

## 3. Inversion double well: tunnelling doublets

In [ ]:
m = rd.compile_inversion(n_basis=80, basis_scale=0.6, potential_order=4)
depths = np.linspace(2, 14, 13)
split = []
for d in depths:
    sol = rd.solve(m, {"G": 1.0, "V2": -d, "V4": 1.0}, nlowest=2, method="dense")
    e = {b["irrep"]: r["eigenvalues"] for b, r in zip(sol.blocks, sol.results)}
    split.append((e["-"][0] - e["+"][0], e["-"][1] - e["+"][1]))
split = np.array(split)
print("ground / first-excited tunnelling splittings vs barrier depth V2:")
for d, (s0, s1) in zip(depths[::3], split[::3]):
    print(f"  V2 = {d:5.1f}: {s0:.3e}  {s1:.3e}   (barrier height {d*d/4:.1f})")
if plt is not None:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.semilogy(depths**2 / 4, split[:, 0], "o-", label="v = 0 doublet")
    ax.semilogy(depths**2 / 4, split[:, 1], "s-", label="v = 1 doublet")
    ax.set(xlabel="barrier height V2^2 / (4 V4)", ylabel="tunnelling splitting", title="Inversion doublets")
    ax.legend(); fig.tight_layout(); fig.savefig(WORK / "05_inversion.png", dpi=150); print("saved", WORK / "05_inversion.png")

## 4. Rho-axis internal rotation in an asymmetric rotor (K mixed)

In [ ]:
F, V3 = 5.3, 370.0
ram = rd.compile_ram_torsion(n_fold=3, m_max=18, j_max=4, d_ab=True)
for rho in (0.0, 0.05, 0.1):
    th = {"A": 4.25 + F * rho**2, "B": 0.82, "C": 0.79, "D_ab": 0.0, "F": F, "D_pa": -2 * F * rho, "V3": V3}
    sol = rd.solve(ram, th, nlowest=3, method="dense")
    e = {(b["two_j"] // 2, b["irrep"]): r["eigenvalues"] for b, r in zip(sol.blocks, sol.results)}
    print(f"rho={rho:4.2f}  J=4 lowest A/E levels:", np.round(e[(4, "sigma=0")][:3], 4), np.round(e[(4, "sigma=1")][:3], 4),
          " certified", sol.certified)